---슈도 코드
모델 지정  
Langchain으로 크로마db 연결  
질문이 들어오면 RAG의 벡터 DB에서 유사도 검색으로 답변  
답변할때 RAG의 근거 데이터 row를 반환  

시간이 남으면 질문 입력 받을때 강아지의 나이를 넣을 내용  
생후진료처 기준 아기견(~2), 성견(2~6), 노령견(7~)으로 바뀌게  
검색 정확도를 높임  

In [5]:
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI


# 노트북 실행 위치가 프로젝트 루트이거나 notebooks 폴더인 경우 모두 지원합니다.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").is_dir() and (PROJECT_DIR.parent / "data").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

load_dotenv(PROJECT_DIR / ".env")
CHROMA_DIR = PROJECT_DIR / "data" / "chroma_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    encode_kwargs={"normalize_embeddings": True},
)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

print(f"프로젝트 경로: {PROJECT_DIR}")
print(f"ChromaDB 파일: {CHROMA_DIR / 'chroma.sqlite3'}")
print("OpenAI 답변 생성 모델 준비 완료")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3844.14it/s]


프로젝트 경로: c:\Users\rhksa\OneDrive\사진\바탕 화면\mle-01-p1-team2
ChromaDB 파일: c:\Users\rhksa\OneDrive\사진\바탕 화면\mle-01-p1-team2\data\chroma_db\chroma.sqlite3
OpenAI 답변 생성 모델 준비 완료


In [6]:
vector_db = Chroma(
    collection_name="pet_care",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

collection_count = vector_db._collection.count()
if collection_count == 0:
    raise RuntimeError(
        f"기존 Chroma 컬렉션 'pet_care'가 비어 있습니다: {CHROMA_DIR / 'chroma.sqlite3'}"
    )

print(f"기존 Chroma 컬렉션 문서 수: {collection_count:,}")


기존 Chroma 컬렉션 문서 수: 100


In [7]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """아래 [검색 데이터]를 근거로 사용자의 질문에 답하세요.
규칙:
1. 검색된 데이터에 근거해서만 답변하세요.
2. 데이터에 없는 내용은 임의로 추측하지 마세요.
3. 답변은 간결하게 작성하세요.

[검색 데이터]
{context}
""",
    ),
    ("human", "{question}"),
])
rag_chain = prompt | model | parser if model is not None else None


In [8]:
def ask_rag(question, k=3):
    docs = vector_db.similarity_search(question, k=k)
    context = "\n\n".join(
        f"질문: {doc.page_content}\n답변: {doc.metadata.get('qa.output', '')}"
        for doc in docs
    )

    if rag_chain is None:
        answer = "유사도 검색은 성공했습니다. 답변 생성에는 OPENAI_API_KEY가 필요합니다."
    else:
        answer = rag_chain.invoke({"context": context, "question": question})

    evidence_rows = [doc.metadata for doc in docs]
    return {"answer": answer, "evidence_rows": evidence_rows}


In [9]:
result = ask_rag("노령견이 가장 위험한 질병이 어떤거야?") 
print("답변:") 
print(result["answer"]) 

print("\n근거 데이터 row:") 
for row in result["evidence_rows"]: 
    print(row)

답변:
노령견에게 가장 위험한 질병 중 하나는 쿠싱 증후군입니다. 이 질병은 호르몬 불균형으로 인해 발생하며, 다뇨, 배가 부풀어 오르는 증상, 피부 문제 등이 나타날 수 있습니다. 또한, 심장 질환, 신장 질환, 암 등도 노령견에서 흔히 발생하는 위험한 질병입니다. 각 질병은 개별적으로 심각할 수 있으므로, 정기적인 검진과 조기 진단이 중요합니다.

근거 데이터 row:
{'qa.instruction': '너는 반려견 치료 경험이 풍부한 전문가야. 보호자가 하는 질문에 대해 성실히 답해줘.', 'meta.department': '내과', 'Unnamed: 0': '76', 'qa.output': '걱정이 많이 되실 것이라 생각됩니다. 현재 말씀주신 증상들은 가벼운 문제로 간주하기에는 적합하지 않을 것 같습니다. 그러나 떨거나 헥헥거리는 증상만으로는 정확한 판단을 내리기 어려운 점을 말씀드립니다. 이러한 증상은 심혈관 질환, 호르몬 문제, 신경계 문제, 소화기 문제 등 여러 가지 원인으로 인해 발생할 수 있습니다. 따라서 지체하지 마시고 병원에 방문하시어 정확한 원인을 파악하는 것이 중요할 것으로 생각됩니다.', 'meta.disease': '기타', 'meta.lifeCycle': '노령견'}
{'Unnamed: 0': '37', 'qa.output': '강아지가 켁켁거리는 증상은 기침의 일종으로 볼 수 있습니다. 드물게 이물이 목에 걸려서 이러한 증상이 나타날 수 있으나, 켁켁거리는 증상에 있어 구체적으로 몇 가지 원인을 고려해 볼 필요가 있습니다. 첫째로, 감기가 원인일 수 있습니다. 이 경우에 콧물이나 재채기와 함께 증상이 나타나는지 확인해 보시길 바랍니다. 둘째로, 기관의 협착이 원인일 수 있으며, 이 경우 콧물이나 재채기가 동반되지 않는 기침 형태로 나타나곤 합니다. 셋째로, 심장 질환으로 인한 폐수종이 가능성이 있으며, 이 경우 운동 능력이 저하되고 운동 후 기침이 심해질 수 있습니다. 네번째로, 이물질이 원인으로 의심될 수 있는 상황입니다. 이와